# EV Charger Anomaly Detection — Exploratory Data Analysis

This notebook explores `charging_logs.csv` and motivates the three-layer detection architecture:
1. **Layer 1** — deterministic physics rules (station-independent impossibilities)
2. **Layer 2** — Isolation Forest on station-relative event features
3. **Layer 3** — LightGBM session/station concurrent risk scoring

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from charger_anomaly.config import FAULT_MESSAGES, OK_REF_PREFIX
from charger_anomaly.preprocessing import coerce_types, load_data
from charger_anomaly.rules import apply_all_rules, get_rule_breakdown
from charger_anomaly.features import add_physics_ratios

sns.set_theme(style="whitegrid")
DATA_PATH = PROJECT_ROOT / "data" / "charging_logs.csv"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_data(DATA_PATH)
df = coerce_types(df)
print(f"Rows: {len(df):,}")
print(f"Stations: {df['station_id'].nunique()}")
print(f"Sessions: {df['session_id'].nunique()}")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")
df.head()

## Dataset structure and missing values

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

events_per_session = df.groupby("session_id").size()
print(events_per_session.describe())

## Station heterogeneity

Global thresholds fail because stations operate at different baselines (e.g., STATION_6 ~246V vs STATION_8 ~226V). Station-relative z-scores are mandatory for Layer 2.

In [ ]:
station_summary = df.groupby("station_id").agg(
    mean_voltage=("voltage", "mean"),
    mean_power=("power_kw", "mean"),
    mean_temp=("temperature_c", "mean"),
    n_events=("session_id", "size"),
).sort_values("mean_voltage")

station_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
station_summary["mean_voltage"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Mean Voltage by Station")
ax.set_ylabel("Volts")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_station_mean_voltage.png", dpi=150)
plt.show()

## Three anomaly populations

| Set | Definition | Role |
|-----|------------|------|
| A — Known faults | `error_code ≠ 0` or explicit fault message | IF threshold tuning; recall headline |
| B — OK (ref=...) | Subtle planted anomalies | Third eval set only |
| C — Physics violations | Layer 1 rules fire | High-confidence deterministic flags |

In [ ]:
known_fault = (df["error_code"] != 0) | df["message"].isin(FAULT_MESSAGES)
ok_ref = df["message"].str.startswith(OK_REF_PREFIX, na=False)

work = add_physics_ratios(df)
physics_violation = apply_all_rules(work)

print(f"Set A — known faults: {known_fault.sum():,}")
print(f"Set B — OK (ref=...): {ok_ref.sum():,}")
print(f"Set C — physics violations: {physics_violation.sum():,}")
print(f"Overlap A ∩ C: {(known_fault & physics_violation).sum():,}")
print(f"Overlap A ∩ B: {(known_fault & ok_ref).sum():,}")

## Layer 1 rule breakdown

In [ ]:
breakdown = get_rule_breakdown(work)
rule_counts = breakdown.sum().sort_values(ascending=False)
rule_counts.plot(kind="barh", figsize=(8, 4), title="Layer 1 Rule Hits")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_layer1_rule_breakdown.png", dpi=150)
plt.show()

## Session-level implicit labels

953/4000 sessions contain at least one known fault. Fault events are uniformly distributed across session position — no within-session early-warning signal. Layer 3 is reframed as concurrent session/station risk scoring.

In [ ]:
session_fault = df.assign(fault=known_fault.astype(int)).groupby("session_id")["fault"].max()
print(f"Fault sessions: {session_fault.sum()} / {len(session_fault)} ({100*session_fault.mean():.1f}%)")

work = work.assign(fault=known_fault, event_idx=work.groupby("session_id").cumcount(),
                   session_size=work.groupby("session_id")["session_id"].transform("size"))
work["event_position"] = work["event_idx"] / (work["session_size"] - 1).clip(lower=1)

fig, ax = plt.subplots(figsize=(8, 4))
work[work["fault"] == 1]["event_position"].hist(bins=20, alpha=0.6, label="fault events", ax=ax)
work[work["fault"] == 0]["event_position"].hist(bins=20, alpha=0.4, label="normal events", ax=ax)
ax.set_xlabel("Event position in session (0=start, 1=end)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_fault_position_uniformity.png", dpi=150)
plt.show()

## Distributions of key telemetry fields

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["voltage", "power_kw", "temperature_c", "current"]):
    sns.histplot(df[col], bins=50, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_telemetry_distributions.png", dpi=150)
plt.show()